In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
list_files = os.listdir("../")
list_files

['.git',
 'away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [3]:
home_score_df = pd.read_csv("../home_team_score.csv")
away_score_df = pd.read_csv("../away_team_score.csv")
event_df = pd.read_csv("../event.csv")
home_team_df = pd.read_csv("../home_team.csv")
away_team_df = pd.read_csv("../away_team.csv")
tournament_df = pd.read_csv("../tournament.csv")


In [4]:

# THRESHOLDS
# ATP/WTA rank valid range: 1–2000 (active ranked players)
# tennis_points floor: 1 (avoid zero-weight tournaments)

MAX_RANK              = 2000
MIN_RANK              = 1
MIN_TOURNAMENT_POINTS = 1      # floor to avoid division/zero-weight issues
DEFAULT_TOURN_POINTS  = 1      # fallback if tennis_points is null
UPSET_SMOOTHING       = 1      # log(1 + upset_factor) smoothing constant

report         = []
total_original = len(event_df)

def log(step, desc, removed, note=''):
    report.append({
        'Step'          : step,
        'Description'   : desc,
        'Rows Removed'  : removed,
        '% of Original' : round(removed / total_original * 100, 2),
        'Note'          : note
    })


# STEP 1: Start from event_df — only finished matches with valid winner_code
# winner_code: 1 = home wins, 2 = away wins

df_event = event_df.copy()
before   = len(df_event)
df_event = df_event[df_event['winner_code'].isin([1, 2])].copy()
removed  = before - len(df_event)
log(1, 'Matches with null or invalid winner_code (unfinished/corrupt)', removed, '')


# STEP 2: Exclude walkovers / retirements using final_result_only flag
# final_result_only=True means the result was recorded without a real match

before   = len(df_event)
if 'final_result_only' in df_event.columns:
    df_event = df_event[df_event['final_result_only'] != True].copy()
removed  = before - len(df_event)
log(2, 'Walkovers/retirements (final_result_only=True)', removed, '')

valid_match_ids = set(df_event['match_id'])


# STEP 3: Load and clean home/away player info
# Drop duplicates, handle corrupt duplicate match_ids

def clean_player_table(df, side_label):
    df = df.copy()

    # Exact duplicate rows
    before = len(df)
    df     = df.drop_duplicates()
    dupes_removed = before - len(df)

    # Only keep valid match_ids
    df = df[df['match_id'].isin(valid_match_ids)].copy()

    # Handle duplicate match_ids — for player tables, one match_id = one player
    # Multiple rows = corrupt (no interrupted match logic needed here)
    before      = len(df)
    df          = df[~df['match_id'].duplicated(keep=False)].copy()
    id_removed  = before - len(df)

    print(f"  [{side_label}] Exact dupes removed: {dupes_removed} | "
          f"Corrupt duplicate match_ids removed: {id_removed}")
    return df

print("\nPlayer table cleaning:")
home_players = clean_player_table(home_team_df, 'HOME')
away_players = clean_player_table(away_team_df, 'AWAY')


# STEP 4: Only keep match_ids present in BOTH player tables AND event_df

before       = len(df_event)
common_ids   = (
    set(home_players['match_id'])
    .intersection(set(away_players['match_id']))
    .intersection(valid_match_ids)
)
df_event     = df_event[df_event['match_id'].isin(common_ids)].copy()
home_players = home_players[home_players['match_id'].isin(common_ids)].copy()
away_players = away_players[away_players['match_id'].isin(common_ids)].copy()
removed      = before - len(df_event)
log(4, 'Matches missing from home or away player table', removed, '')


# STEP 5: Validate player ranks

before = len(df_event)

# Null out invalid ranks in player tables
for df_, col in [(home_players, 'current_rank'), (away_players, 'current_rank')]:
    invalid = (
        df_[col].isna() |
        (df_[col] < MIN_RANK) |
        (df_[col] > MAX_RANK) |
        (df_[col] == 0)
    )
    df_.loc[invalid, col] = np.nan

# Drop matches where either player rank is missing
# We cannot compute upset factor without both ranks
home_no_rank = set(home_players[home_players['current_rank'].isna()]['match_id'])
away_no_rank = set(away_players[away_players['current_rank'].isna()]['match_id'])
bad_rank_ids = home_no_rank.union(away_no_rank)

df_event     = df_event[~df_event['match_id'].isin(bad_rank_ids)].copy()
home_players = home_players[~home_players['match_id'].isin(bad_rank_ids)].copy()
away_players = away_players[~away_players['match_id'].isin(bad_rank_ids)].copy()
removed      = before - len(df_event)
log(5, f'Matches with null/invalid ranks (outside {MIN_RANK}–{MAX_RANK} or zero)',
    removed, '')


# STEP 6: Validate country field — drop null or empty countries

before = len(df_event)
for df_ in [home_players, away_players]:
    df_['country'] = df_['country'].str.strip().str.upper()
    df_.loc[df_['country'].isin(['', 'N/A', 'NA', 'NONE', 'UNKNOWN']), 'country'] = np.nan

no_country_home = set(home_players[home_players['country'].isna()]['match_id'])
no_country_away = set(away_players[away_players['country'].isna()]['match_id'])
bad_country_ids = no_country_home.union(no_country_away)

df_event     = df_event[~df_event['match_id'].isin(bad_country_ids)].copy()
home_players = home_players[~home_players['match_id'].isin(bad_country_ids)].copy()
away_players = away_players[~away_players['match_id'].isin(bad_country_ids)].copy()
removed      = before - len(df_event)
log(6, 'Matches with null/invalid country', removed, '')


# STEP 7: Validate gender — split ATP (M) and WTA (F) to avoid cross-ranking

before = len(df_event)

# Flag matches where home and away genders differ (corrupt data)
home_gender   = home_players.set_index('match_id')['gender']
away_gender   = away_players.set_index('match_id')['gender']
common_gender = home_gender.index.intersection(away_gender.index)
gender_clash  = set(
    common_gender[home_gender.loc[common_gender] != away_gender.loc[common_gender]]
)

df_event     = df_event[~df_event['match_id'].isin(gender_clash)].copy()
home_players = home_players[~home_players['match_id'].isin(gender_clash)].copy()
away_players = away_players[~away_players['match_id'].isin(gender_clash)].copy()
removed      = before - len(df_event)
log(7, 'Matches with gender mismatch between home and away player', removed,
    'Prevents comparing ATP and WTA rankings')


# STEP 8: Validate same player appearing on both sides (impossible in singles)

before       = len(df_event)
home_pid     = home_players.set_index('match_id')['player_id']
away_pid     = away_players.set_index('match_id')['player_id']
common_pid   = home_pid.index.intersection(away_pid.index)
same_player  = set(common_pid[home_pid.loc[common_pid] == away_pid.loc[common_pid]])

df_event     = df_event[~df_event['match_id'].isin(same_player)].copy()
home_players = home_players[~home_players['match_id'].isin(same_player)].copy()
away_players = away_players[~away_players['match_id'].isin(same_player)].copy()
removed      = before - len(df_event)
log(8, 'Matches where home and away player_id are identical', removed, '')


# STEP 9: Load tournament info and validate tennis_points

tourn_clean = tournament_df.drop_duplicates(subset='match_id').copy()
tourn_clean = tourn_clean[tourn_clean['match_id'].isin(
    set(df_event['match_id'])
)][['match_id', 'tennis_points', 'competition_type', 'ground_type']].copy()

# Null out invalid tournament points
tourn_clean.loc[
    tourn_clean['tennis_points'].isna() | (tourn_clean['tennis_points'] < 0),
    'tennis_points'
] = np.nan

# Apply floor to avoid zero-weight
tourn_clean['tennis_points'] = tourn_clean['tennis_points'].clip(
    lower=MIN_TOURNAMENT_POINTS
)

# Fill missing tournament points with default
missing_tourn = tourn_clean['tennis_points'].isna().sum()
tourn_clean['tennis_points'] = tourn_clean['tennis_points'].fillna(DEFAULT_TOURN_POINTS)

report.append({
    'Step'          : 9,
    'Description'   : 'Missing tournament points filled with default weight',
    'Rows Removed'  : f'{missing_tourn} filled',
    '% of Original' : '-',
    'Note'          : f'Default = {DEFAULT_TOURN_POINTS} (minimum weight)'
})


# STEP 10: Assemble match-level dataframe

match_df = df_event[['match_id', 'winner_code']].copy()
match_df = match_df.merge(
    home_players[['match_id', 'player_id', 'current_rank', 'country', 'gender']]
    .rename(columns={'player_id'    : 'home_player_id',
                     'current_rank' : 'home_rank',
                     'country'      : 'home_country',
                     'gender'       : 'gender'}),
    on='match_id', how='inner'
)
match_df = match_df.merge(
    away_players[['match_id', 'player_id', 'current_rank', 'country']]
    .rename(columns={'player_id'    : 'away_player_id',
                     'current_rank' : 'away_rank',
                     'country'      : 'away_country'}),
    on='match_id', how='inner'
)
match_df = match_df.merge(
    tourn_clean[['match_id', 'tennis_points', 'ground_type']],
    on='match_id', how='left'
)
match_df['tennis_points'] = match_df['tennis_points'].fillna(DEFAULT_TOURN_POINTS)


# SCORING FORMULA
#
# upset_factor = winner_rank / loser_rank
#   > 1  → upset  (lower-ranked beat higher-ranked) → high score
#   < 1  → expected win                             → low score
#   = 1  → same rank                                → neutral
#
# match_score = log(1 + upset_factor) × log(1 + tournament_weight)
#   log smoothing on both sides prevents extreme outliers dominating

match_df['winner_rank'] = np.where(
    match_df['winner_code'] == 1,
    match_df['home_rank'],
    match_df['away_rank']
)
match_df['loser_rank'] = np.where(
    match_df['winner_code'] == 1,
    match_df['away_rank'],
    match_df['home_rank']
)
match_df['winner_country'] = np.where(
    match_df['winner_code'] == 1,
    match_df['home_country'],
    match_df['away_country']
)
match_df['winner_player_id'] = np.where(
    match_df['winner_code'] == 1,
    match_df['home_player_id'],
    match_df['away_player_id']
)

match_df['upset_factor'] = match_df['winner_rank'] / match_df['loser_rank']
match_df['match_score']  = (
    np.log1p(match_df['upset_factor']) *
    np.log1p(match_df['tennis_points'])
)


# AGGREGATE — per player first, then per country

player_scores = (
    match_df.groupby(['winner_player_id', 'winner_country', 'gender'])
    .agg(
        total_score   = ('match_score',  'sum'),
        total_wins    = ('match_id',     'count'),
        upset_wins    = ('upset_factor', lambda x: (x > 1).sum()),
        avg_score     = ('match_score',  'mean'),
        best_win_rank = ('loser_rank',   'min'),   # lowest rank beaten = best win
    )
    .reset_index()
)

country_scores = (
    player_scores.groupby(['winner_country', 'gender'])
    .agg(
        total_score      = ('total_score',   'sum'),
        total_wins       = ('total_wins',    'sum'),
        upset_wins       = ('upset_wins',    'sum'),
        active_players   = ('winner_player_id', 'nunique'),
        avg_score_player = ('total_score',   'mean'),
        best_win_rank    = ('best_win_rank', 'min'),
    )
    .reset_index()
    .sort_values('total_score', ascending=False)
)




Player table cleaning:
  [HOME] Exact dupes removed: 7710 | Corrupt duplicate match_ids removed: 10417
  [AWAY] Exact dupes removed: 7322 | Corrupt duplicate match_ids removed: 9797


In [5]:

total_clean   = len(match_df)
total_removed = total_original - total_clean

print("\n" + "=" * 70)
print("DATA CLEANING REPORT — Country Success Analysis")
print("=" * 70)
print(f"  Original match rows  : {total_original:,}")
print(f"  Matches after clean  : {total_clean:,}")
print(f"  Total removed        : {total_removed:,} ({round(total_removed/total_original*100,2)}%)")
print(f"  Clean data           : {round(total_clean/total_original*100,2)}%")
print("=" * 70)
print(pd.DataFrame(report).to_string(index=False))



DATA CLEANING REPORT — Country Success Analysis
  Original match rows  : 35,053
  Matches after clean  : 9,464
  Total removed        : 25,589 (73.0%)
  Clean data           : 27.0%
 Step                                                   Description Rows Removed % of Original                                    Note
    1 Matches with null or invalid winner_code (unfinished/corrupt)         3049           8.7                                        
    2                Walkovers/retirements (final_result_only=True)         2196          6.26                                        
    4                Matches missing from home or away player table        20175         57.56                                        
    5      Matches with null/invalid ranks (outside 1–2000 or zero)          135          0.39                                        
    6                             Matches with null/invalid country            6          0.02                                        
    7  

In [6]:

# Results by gender
for gender_val, gender_label in [('M', 'ATP — MEN'), ('F', 'WTA — WOMEN')]:
    subset = country_scores[country_scores['gender'] == gender_val].head(3)
    if len(subset) == 0:
        continue

    print(f"\n{'=' * 70}")
    print(f"TOP 15 COUNTRIES — {gender_label}")
    print(f"{'=' * 70}")
    print(f"  Scoring: match_score = log(1 + winner_rank/loser_rank) × log(1 + tournament_points)")
    print(f"  Higher score = more upset wins in more prestigious tournaments")
    print(f"{'=' * 70}")
    print(f"\n  {'Rank':<5} {'Country':<10} {'Score':>10} {'Wins':>7} "
          f"{'Upsets':>8} {'Players':>8} {'Best Win (Rank)':>16}")
    print(f"  {'-'*5} {'-'*10} {'-'*10} {'-'*7} {'-'*8} {'-'*8} {'-'*16}")

    for i, (_, row) in enumerate(subset.iterrows(), 1):
        print(f"  {i:<5} {row['winner_country']:<10} "
              f"{row['total_score']:>10.2f} "
              f"{int(row['total_wins']):>7,} "
              f"{int(row['upset_wins']):>8,} "
              f"{int(row['active_players']):>8} "
              f"#{int(row['best_win_rank']):>14}")


TOP 15 COUNTRIES — ATP — MEN
  Scoring: match_score = log(1 + winner_rank/loser_rank) × log(1 + tournament_points)
  Higher score = more upset wins in more prestigious tournaments

  Rank  Country         Score    Wins   Upsets  Players  Best Win (Rank)
  ----- ---------- ---------- ------- -------- -------- ----------------
  1     USA            663.55     394      148       76 #             9
  2     FRANCE         647.35     562      182       97 #             4
  3     ARGENTINA      510.70     315      127       53 #            22

TOP 15 COUNTRIES — WTA — WOMEN
  Scoring: match_score = log(1 + winner_rank/loser_rank) × log(1 + tournament_points)
  Higher score = more upset wins in more prestigious tournaments

  Rank  Country         Score    Wins   Upsets  Players  Best Win (Rank)
  ----- ---------- ---------- ------- -------- -------- ----------------
  1     USA            795.80     406      162       76 #             2
  2     RUSSIA         545.35     331      129       5